# Notebook 10 — Demonstração Ponta a Ponta da Decisão

**Objetivo**: mostrar em um único notebook, sem servidor externo, os quatro elementos
exigidos pela evidência de aceite da Etapa 5:

| # | Elemento | Onde aparece |
|---|----------|--------------|
| 1 | Braço selecionado | `response["arm_id"]` / `arm_name` |
| 2 | Justificativa estatística | `response["reason_codes"]` |
| 3 | Versão da política | `response["policy_version"]` |
| 4 | Registro auditável | entrada gerada em `decision_log.jsonl` |

O ciclo completo é: **decide → reward → ver log**.

In [1]:
import sys, os, json, uuid, tempfile
from pathlib import Path
from datetime import datetime

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from datathon_offerexp.policies import ThompsonSamplingPolicy, OFFER_CATALOG
from datathon_offerexp.decision_log import DecisionLog
from datathon_offerexp.evaluation import replay_evaluate
import pandas as pd

SEED = 42
EVENTS_PATH = Path('..') / 'data' / 'synthetic_enrichment' / 'offer_events.csv'
LOG_PATH = Path(tempfile.mktemp(suffix='_decision_log.jsonl'))

print('Imports OK')
print(f'Log temporário: {LOG_PATH}')

Imports OK
Log temporário: /tmp/tmpy7o7nhy9_decision_log.jsonl


## 1. Aquecimento da política

A política é treinada no histórico de eventos via Replayer Method —
exatamente como em produção após um deploy inicial.

In [2]:
policy = ThompsonSamplingPolicy(seed=SEED)
events = pd.read_csv(EVENTS_PATH)
replay_evaluate(policy, events, seed=SEED)

stats = policy.stats()
print(f'Política: {policy.version}')
print(f'Total de rounds de treino: {sum(s["trials"] for s in stats)}')
print()
print(f'{"Braço":<25} {"trials":>7} {"alpha":>7} {"beta":>7} {"reward_rate":>12}')
print('-' * 60)
for s in stats:
    print(f'{s["arm_name"]:<25} {s["trials"]:>7} {s.get("alpha",1):>7.1f} {s.get("beta",1):>7.1f} {s["reward_rate"]:>12.4f}')

Política: thompson-v1
Total de rounds de treino: 10172

Braço                      trials   alpha    beta  reward_rate
------------------------------------------------------------
sem_oferta                    381     9.0   374.0       0.0210
educacao_financeira          4434   836.0  3600.0       0.1883
simulador_credito            2697   139.0  2560.0       0.0512
cartao_premium               2660   149.0  2513.0       0.0556


## 2. Decisão — equivalente a POST /decide

Simula a chamada com um contexto de cliente.
A resposta inclui os quatro elementos da evidência de aceite.

In [3]:
log = DecisionLog(str(LOG_PATH))

event_id    = f'demo-{uuid.uuid4().hex[:8]}'
subject_key = 'cliente-banca-demo'
context = {
    'idade': 35,
    'profissao': 'engenheiro',
    'escolaridade': 'superior',
    'numero_contatos_campanha': 2,
}

arm_idx = policy.select_arm()
arm = OFFER_CATALOG[arm_idx]

s = next(x for x in policy.stats() if x['arm_id'] == arm_idx)
reason_codes = [
    f'thompson_sample_arm_{arm_idx}',
    f'alpha={s.get("alpha", 1):.1f}',
    f'beta={s.get("beta", 1):.1f}',
    f'reward_rate={s["reward_rate"]:.4f}',
    f'trials={s["trials"]}',
]

decision_id = log.log(
    event_id=event_id,
    arm_id=arm_idx,
    arm_name=arm['arm_name'],
    reward=None,
    policy_version=policy.version,
    reason_codes=reason_codes,
)

response = {
    'event_id':       event_id,
    'arm_id':         arm_idx,
    'arm_name':       arm['arm_name'],
    'policy_version': policy.version,
    'reason_codes':   reason_codes,
    'decided_at':     datetime.utcnow().isoformat(),
}

print('=' * 60)
print('  RESPOSTA — POST /decide')
print('=' * 60)
print(json.dumps(response, indent=2, ensure_ascii=False))

  RESPOSTA — POST /decide
{
  "event_id": "demo-2a200f0e",
  "arm_id": 1,
  "arm_name": "educacao_financeira",
  "policy_version": "thompson-v1",
  "reason_codes": [
    "thompson_sample_arm_1",
    "alpha=836.0",
    "beta=3600.0",
    "reward_rate=0.1883",
    "trials=4434"
  ],
  "decided_at": "2026-06-17T04:06:47.381748"
}


## 3. Recompensa — equivalente a POST /reward

Registra a recompensa após a interação do cliente.
O prior Beta é atualizado e o log é completado.

In [4]:
reward_value = 1.0

policy.update(arm_idx, reward_value)

log.log(
    event_id=event_id,
    arm_id=arm_idx,
    arm_name=arm['arm_name'],
    reward=reward_value,
    policy_version=policy.version,
    reason_codes=['reward_registered'],
)

s_after = next(x for x in policy.stats() if x['arm_id'] == arm_idx)
print(f'Recompensa {reward_value} registrada para arm_id={arm_idx} ({arm["arm_name"]})')
print()
print(f'Prior atualizado:  alpha={s_after.get("alpha",1):.1f}  beta={s_after.get("beta",1):.1f}  reward_rate={s_after["reward_rate"]:.4f}')

Recompensa 1.0 registrada para arm_id=1 (educacao_financeira)

Prior atualizado:  alpha=837.0  beta=3600.0  reward_rate=0.1885


## 4. Registro auditável

O arquivo `decision_log.jsonl` contém uma linha JSON por evento.
Cada linha é imutável (append-only) e inclui `decision_id`, `arm_id`,
`reason_codes`, `policy_version` e `timestamp`.

In [5]:
records = log.read_all()

print(f'Entradas geradas nesta sessão: {len(records)}')
print()
for i, rec in enumerate(records, 1):
    print(f'─── Entrada {i} ───────────────────────────────────────')
    print(json.dumps(rec, indent=2, ensure_ascii=False))
    print()

Entradas geradas nesta sessão: 2

─── Entrada 1 ───────────────────────────────────────
{
  "decision_id": "86a97bc7-8af1-4b19-89bd-b5252d5e8645",
  "event_id": "demo-2a200f0e",
  "policy_version": "thompson-v1",
  "arm_id": 1,
  "arm_name": "educacao_financeira",
  "reward": null,
  "reason_codes": [
    "thompson_sample_arm_1",
    "alpha=836.0",
    "beta=3600.0",
    "reward_rate=0.1883",
    "trials=4434"
  ],
  "timestamp": "2026-06-17T04:06:47.381153"
}

─── Entrada 2 ───────────────────────────────────────
{
  "decision_id": "1352da7f-a08e-43f3-99dc-4fcbba64d367",
  "event_id": "demo-2a200f0e",
  "policy_version": "thompson-v1",
  "arm_id": 1,
  "arm_name": "educacao_financeira",
  "reward": 1.0,
  "reason_codes": [
    "reward_registered"
  ],
  "timestamp": "2026-06-17T04:06:47.388046"
}



## Resumo da evidência de aceite

| # | Elemento | Localização | Valor |
|---|----------|-------------|-------|
| 1 | **Braço selecionado** | `response["arm_id"]` / `arm_name` | ver saída da célula 2 |
| 2 | **Justificativa** | `response["reason_codes"]` | alpha, beta, reward_rate, trials |
| 3 | **Versão da política** | `response["policy_version"]` | `thompson-v1` |
| 4 | **Registro auditável** | `decision_log.jsonl` | entradas exibidas na célula 4 |

---

### Via API REST (requer `make api`)

```bash
# Decide
curl -s -X POST http://localhost:8000/decide \\
  -H "Content-Type: application/json" \\
  -d '{"event_id":"demo-001","subject_key":"cliente-banca","context":{"idade":35}}'

# Registra recompensa (use arm_id retornado acima)
curl -s -X POST http://localhost:8000/reward \\
  -H "Content-Type: application/json" \\
  -d '{"event_id":"demo-001","arm_id":2,"reward":1.0}'
```